# Model Training - Custom NER Model Eğitimi

Bu notebook, CoNLL formatındaki verilerle custom NER modeli eğitir.

## Adımlar:
1. **Setup & License** - Spark NLP Healthcare lisansı ve ortam kurulumu
2. **CoNLL Dataset Yükleme** - Eğitim için CoNLL verisini yükleme
3. **Train/Validation Split** - Veriyi eğitim ve validasyon setlerine ayırma
4. **Model Eğitimi** - Custom NER modeli eğitimi
5. **Model Değerlendirme** - Precision, Recall, F1-score metrikleri
6. **Model Kaydetme** - Eğitilmiş modeli kaydetme

**Gereksinimler:**
- `1_data_preparation.ipynb` notebook'unun çalıştırılmış olması
- `data/conll/conll2003_text_file.conll` dosyasının mevcut olması


## 1. Setup & License Configuration


In [1]:
import json
import os
from pathlib import Path

# License dosyasını yükle (Kaggle/Colab için)
# Kaggle için:
try:
    with open(r'C:\huseyin\john_snow_labs\generating_conll_files_from_pretrained_models\spark_jsl.json') as f:
        license_keys = json.load(f)
except:
    # Colab için:
    try:
        from google.colab import files
        if 'spark_jsl.json' not in os.listdir():
            print("Please upload your spark_jsl.json license file:")
            uploaded = files.upload()
            os.rename(list(uploaded.keys())[0], 'spark_jsl.json')
        with open('spark_jsl.json') as f:
            license_keys = json.load(f)
    except:
        # Local için:
        with open('spark_jsl.json') as f:
            license_keys = json.load(f)

# License key'leri environment variable olarak ayarla
locals().update(license_keys)
os.environ.update(license_keys)

print("✅ License keys loaded")
print(f"JSL Version: {license_keys.get('JSL_VERSION', 'N/A')}")
print(f"Public Version: {license_keys.get('PUBLIC_VERSION', 'N/A')}")


✅ License keys loaded
JSL Version: 6.1.1
Public Version: 6.1.3


In [2]:
# Kütüphaneleri yükle (eğer yüklenmemişse)
import subprocess

try:
    import torch
    gpu_available = torch.cuda.is_available()
except:
    gpu_available = False
    if gpu_available:
        %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
    else:
        %pip install -q torch torchvision torchaudio

%pip install --upgrade -q pyspark==3.4.1 spark-nlp==$PUBLIC_VERSION
%pip install --upgrade -q spark-nlp-jsl==$JSL_VERSION --extra-index-url https://pypi.johnsnowlabs.com/$SECRET
%pip install -q spark-nlp-display pandas numpy

print("✅ Libraries installed")


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
✅ Libraries installed


In [ ]:
# Spark Session başlat
import sparknlp
import sparknlp_jsl
from pyspark.sql import SparkSession
import os
from pathlib import Path

# GPU kontrolü
try:
    import torch
    gpu_available = torch.cuda.is_available()
except:
    gpu_available = False

# Lokal ortam için cache klasörleri
current_dir = Path(os.getcwd())
cache_dir = current_dir.parent / "cache_pretrained"
cache_dir.mkdir(parents=True, exist_ok=True)
cache_path = str(cache_dir.absolute())

# Spark konfigürasyonu - lokal ortam için optimize edilmiş
params = {
    "spark.driver.memory": "8G",  # Lokal için daha düşük
    "spark.kryoserializer.buffer.max": "1000M",
    "spark.driver.maxResultSize": "1000M",
    "spark.sql.execution.arrow.pyspark.enabled": "true",
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
    "spark.driver.extraJavaOptions": "-Xmx6g"
}

if gpu_available:
    params.update({
        "spark.jsl.settings.pretrained.cache_folder": cache_path,
        "spark.jsl.settings.storage.cluster_tmp_dir": cache_path,
        "spark.jsl.settings.annotator.gpu": "true"
    })
    print("✅ GPU acceleration enabled")
else:
    # CPU için cache ayarları
    params.update({
        "spark.jsl.settings.pretrained.cache_folder": cache_path,
        "spark.jsl.settings.storage.cluster_tmp_dir": cache_path
    })

# Spark session başlat - hata yakalama ile
try:
    print("Starting Spark session...")
    spark = sparknlp_jsl.start(license_keys['SECRET'], params=params)
    spark.sparkContext.setLogLevel("ERROR")  # Log seviyesini düşür
    
    print(f"✅ Spark NLP Version: {sparknlp.version()}")
    print(f"✅ Spark NLP JSL Version: {sparknlp_jsl.version()}")
    print("✅ Spark session initialized")
    
    # Test et
    test_df = spark.createDataFrame([("test",)], ["text"])
    test_df.show(truncate=False)
    print("✅ Spark session is working correctly")
    
except Exception as e:
    print(f"❌ Error starting Spark session with sparknlp_jsl.start(): {e}")
    print("\nTrying alternative method with SparkSession.builder...")
    
    # Alternatif yöntem: SparkSession.builder kullan
    try:
        builder = (
            SparkSession.builder
            .appName("SparkNLPHealthcare")
            .config("spark.driver.memory", "8G")
            .config("spark.kryoserializer.buffer.max", "1000M")
            .config("spark.driver.maxResultSize", "1000M")
            .config("spark.sql.execution.arrow.pyspark.enabled", "true")
            .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
            .config("spark.driver.extraJavaOptions", "-Xmx6g")
        )
        
        if gpu_available:
            builder = builder.config("spark.jsl.settings.annotator.gpu", "true")
            print("✅ GPU acceleration enabled (alternative method)")
        
        spark = builder.getOrCreate()
        spark.sparkContext.setLogLevel("ERROR")
        
        # sparknlp_jsl'i manuel olarak yapılandır
        spark.conf.set("spark.jsl.settings.pretrained.cache_folder", cache_path)
        spark.conf.set("spark.jsl.settings.storage.cluster_tmp_dir", cache_path)
        
        print(f"✅ Spark NLP Version: {sparknlp.version()}")
        print("✅ Spark session initialized (alternative method)")
        
        # Test et
        test_df = spark.createDataFrame([("test",)], ["text"])
        test_df.show(truncate=False)
        print("✅ Spark session is working correctly")
        
    except Exception as e2:
        print(f"❌ Failed with alternative method: {e2}")
        print("\n⚠️ Please check:")
        print("   1. Java is installed (java -version)")
        print("   2. JAVA_HOME environment variable is set")
        print("   3. Spark NLP JSL packages are properly installed")
        raise


✅ GPU acceleration enabled


RuntimeError: Java gateway process exited before sending its port number

In [ ]:
# Modülleri import et
import sys
import os
from pathlib import Path

# Notebook'un bulunduğu dizini tespit et
current_dir = Path(os.getcwd())

# src dizinini bul
src_paths = [
    # Lokal ortam: notebooks/ klasöründen bir üst dizindeki src
    current_dir.parent / 'src',
    # Lokal ortam: mevcut dizindeki src
    current_dir / 'src',
    # Kaggle/Colab ortamları
    Path('/kaggle/working/src'),
    Path('/content/src'),
    # Relative paths
    Path('../src'),
    Path('src'),
]

src_path = None
for path in src_paths:
    if path.exists() and path.is_dir():
        if str(path.parent) not in sys.path:
            sys.path.insert(0, str(path.parent))
        src_path = path
        print(f"✅ Found src directory at: {path}")
        break

if not src_path:
    print("⚠️ src directory not found in any of the following locations:")
    for path in src_paths:
        print(f"   - {path}")
    print("\nPlease ensure the src directory exists relative to the notebook location.")
else:
    try:
        from src import ModelTrainer
        print("✅ Modules imported successfully")
    except ImportError as e:
        print(f"❌ Import error: {e}")
        print(f"   Make sure src/__init__.py exists and exports ModelTrainer")


## 2. CoNLL Dataset Yükleme


In [ ]:
# Model trainer'ı başlat
trainer = ModelTrainer(spark)

# Embeddings yükle
print("Loading clinical embeddings...")
clinical_embeddings = trainer.load_embeddings("embeddings_clinical")
print("✅ Embeddings loaded")


In [ ]:
# CoNLL dataset'ini yükle
conll_path = "data/conll/conll2003_text_file.conll"

# Dosyanın varlığını kontrol et
if not Path(conll_path).exists():
    print(f"❌ CoNLL file not found: {conll_path}")
    print("Please run 1_data_preparation.ipynb first to generate the CoNLL file.")
else:
    print(f"Loading CoNLL dataset from {conll_path}...")
    training_data = trainer.load_conll_dataset(conll_path)
    print(f"✅ Dataset loaded: {training_data.count()} sentences")
    training_data.show(3)


## 3. Train/Validation Split


In [ ]:
# Dataset'i train ve validation setlerine ayır
train_data, validation_data = trainer.split_dataset(training_data, train_ratio=0.8, seed=100)

# Validation data'yı parquet formatında kaydet (evaluation için)
test_data_path = "data/processed/test_data.parquet"
Path(test_data_path).parent.mkdir(parents=True, exist_ok=True)
clinical_embeddings.transform(validation_data).write.mode("overwrite").parquet(test_data_path)
print(f"✅ Test data saved to {test_data_path}")


## 4. Model Eğitimi


In [ ]:
# GPU için batch size optimizasyonu
try:
    import torch
    use_gpu = torch.cuda.is_available()
    if use_gpu:
        batch_size = 16
        print("🚀 GPU detected - using optimized batch size")
    else:
        batch_size = 8
        print("Using CPU mode - standard batch size")
except:
    batch_size = 8
    use_gpu = False

# Training pipeline oluştur
training_pipeline = trainer.create_training_pipeline(
    max_epochs=20,
    lr=0.003,
    batch_size=batch_size,
    random_seed=0,
    verbose=1,
    test_dataset=test_data_path,
    output_logs_path="./ner_logs",
    validation_split=0.1,
    use_best_model=True,
    early_stopping_criterion=0.04,
    early_stopping_patience=3
)

print("✅ Training pipeline created")
print(f"Training parameters:")
print(f"  - Max epochs: 20")
print(f"  - Learning rate: 0.003")
print(f"  - Batch size: {batch_size} {'(GPU optimized)' if use_gpu else '(CPU)'}")
print(f"  - Validation split: 0.1")
print(f"  - Early stopping: Enabled")


In [ ]:
# Model eğitimi başlat
try:
    import torch
    if torch.cuda.is_available():
        print("🚀 Starting model training with GPU acceleration...")
        print(f"   GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("Starting model training on CPU... This may take several minutes...")
except:
    print("Starting model training... This may take several minutes...")

trained_model = trainer.train_model(train_data, training_pipeline)

print("\n✅ Model training completed!")
print("\nTraining logs saved to ./ner_logs/")


## 5. Model Değerlendirme


In [ ]:
# Model performansını değerlendir
print("Evaluating model performance...")
eval_results = trainer.evaluate_model(validation_data, drop_o=True, case_sensitive=True)

print("\n✅ Evaluation completed!")


## 6. Model Kaydetme


In [ ]:
# Modeli kaydet
model_path = "models/trained/custom_ner_model"
Path(model_path).parent.mkdir(parents=True, exist_ok=True)

trainer.save_model(model_path)
print(f"✅ Model saved to {model_path}")

print("\n✅ Model training pipeline completed!")
print("\nNext step: Run 3_prediction.ipynb to use the trained model for predictions.")
